# **Model Size Comparison**
---

In [1]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from detection_system.loader import load_model
from detection_system.inference import detect_frame
from detection_system.utils import load_config, save_figure, save_table, frames_from_video

cfg = load_config("config.yaml")
plt.style.use(cfg["figures"]["style"])
CONF = cfg["model"]["confidence_threshold"]
print("Setup complete")

Setup complete


In [ ]:
# ── Loading benchmark results ────────────────────────────────────────────
BENCH_CSV = PROJECT_ROOT / "data" / "processed" / "benchmark_results.csv"

if not BENCH_CSV.exists():
    raise FileNotFoundError(
        f"{BENCH_CSV} not found. Run Notebook 06 first."
    )

bench_df = pd.read_csv(BENCH_CSV)
print("Loaded benchmark results:")
display(bench_df[["model_name", "model_size_mb", "mean_fps",
                  "mean_latency_ms", "p95_latency_ms", "peak_memory_mb"]])

Loaded benchmark results:


,model_name,model_size_mb,mean_fps,mean_latency_ms,p95_latency_ms,peak_memory_mb
0,YOLOv8-nano,NaN,9.58,104.35,124.30,1.26
1,YOLOv8-small,NaN,4.37,229.06,275.96,4.98
2,YOLOv8-medium,NaN,1.20,831.70,1388.52,0.48


In [ ]:
# ── Adding parameter count (from Ultralytics model info) ─────────────────

MODELS_DIR = PROJECT_ROOT / "models"

param_counts = {}
variant_map = {
    "YOLOv8-nano":   "nano",
    "YOLOv8-small":  "small",
    "YOLOv8-medium": "medium",
}

for model_name in bench_df["model_name"]:
    variant = variant_map.get(model_name)
    if variant:
        try:
            model = load_model(variant, models_dir=MODELS_DIR)
            # Count trainable parameters
            n_params = sum(p.numel() for p in model.model.parameters())
            param_counts[model_name] = n_params / 1e6  # In millions
            del model
            import gc; gc.collect()
            print(f"  {model_name}: {n_params/1e6:.1f}M parameters")
        except Exception as e:
            print(f"  {model_name}: Could not count params ({e})")
            param_counts[model_name] = float("nan")

bench_df["params_M"] = bench_df["model_name"].map(param_counts)
print("\nParameter counts added to benchmark DataFrame.")

Weights will be saved to: c:\Users\Peter\Documents\projects\AI_research\realtime-object-detection\models
Model loaded: YOLOv8 NANO | Classes: 80
  YOLOv8-nano: 3.2M parameters
Weights will be saved to: c:\Users\Peter\Documents\projects\AI_research\realtime-object-detection\models
Model loaded: YOLOv8 SMALL | Classes: 80
  YOLOv8-small: 11.2M parameters
Weights will be saved to: c:\Users\Peter\Documents\projects\AI_research\realtime-object-detection\models
Model loaded: YOLOv8 MEDIUM | Classes: 80
  YOLOv8-medium: 25.9M parameters

Parameter counts added to benchmark DataFrame.


In [ ]:
# ── Detection count comparison across models ─────────────────────────

real_video = PROJECT_ROOT / "data" / "raw" / "sample_traffic.mp4"
mock_video = PROJECT_ROOT / "data" / "mock" / "mock_video.mp4"
VIDEO_PATH = real_video if real_video.exists() else mock_video

import cv2
test_frames = frames_from_video(VIDEO_PATH, max_frames=20)
test_frames = [cv2.resize(f, (640, 480)) for f in test_frames]

detection_counts = {}
for model_name, variant in variant_map.items():
    if model_name not in bench_df["model_name"].values:
        continue
    model = load_model(variant, models_dir=MODELS_DIR)
    counts = []
    for frame in test_frames:
        dets, _ = detect_frame(model, frame, confidence=CONF)
        counts.append(len(dets))
    detection_counts[model_name] = {
        "mean_det_count": np.mean(counts),
        "std_det_count": np.std(counts),
    }
    del model
    import gc; gc.collect()
    print(f"  {model_name}: mean {np.mean(counts):.1f} ± {np.std(counts):.1f} dets/frame")

for model_name, stats in detection_counts.items():
    idx = bench_df[bench_df["model_name"] == model_name].index
    if len(idx) > 0:
        bench_df.loc[idx, "mean_det_count"] = stats["mean_det_count"]
        bench_df.loc[idx, "std_det_count"] = stats["std_det_count"]

Extracted 20 frames from sample_traffic.mp4
Weights will be saved to: c:\Users\Peter\Documents\projects\AI_research\realtime-object-detection\models
Model loaded: YOLOv8 NANO | Classes: 80
  YOLOv8-nano: mean 7.4 ± 2.5 dets/frame
Weights will be saved to: c:\Users\Peter\Documents\projects\AI_research\realtime-object-detection\models
Model loaded: YOLOv8 SMALL | Classes: 80
  YOLOv8-small: mean 7.0 ± 2.7 dets/frame
Weights will be saved to: c:\Users\Peter\Documents\projects\AI_research\realtime-object-detection\models
Model loaded: YOLOv8 MEDIUM | Classes: 80
  YOLOv8-medium: mean 8.4 ± 3.2 dets/frame


In [ ]:
# ── Multi-metric comparison ──────────────────────

palette = sns.color_palette("colorblind", n_colors=len(bench_df))
color_map = dict(zip(bench_df["model_name"], palette))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel A: FPS vs. Model Size scatter
for _, row in bench_df.iterrows():
    size = row["model_size_mb"]
    fps = row["mean_fps"]
    mem = row["peak_memory_mb"]
    color = color_map[row["model_name"]]
    
    # Dot size = memory usage
    dot_size = max(50, mem * 10) if not np.isnan(mem) else 100
    
    axes[0].scatter(
        size, fps,
        s=dot_size,
        color=color,
        alpha=0.85,
        edgecolors="white",
        linewidths=1.5,
        zorder=5,
    )
    axes[0].annotate(
        row["model_name"].replace("YOLOv8-", ""),
        (size, fps),
        textcoords="offset points",
        xytext=(8, 4),
        fontsize=10,
        fontweight="bold",
        color=color,
    )

axes[0].set_xlabel("Model size (MB)", fontsize=12)
axes[0].set_ylabel("Mean FPS", fontsize=12)
axes[0].set_title("Speed vs. Model Size\n(dot size = peak memory)", fontsize=12, fontweight="bold")
axes[0].grid(alpha=0.4)

# Panel B: Grouped bar — mean detection count
det_counts = bench_df["mean_det_count"].fillna(0)
det_stds = bench_df["std_det_count"].fillna(0)

bars = axes[1].bar(
    bench_df["model_name"],
    det_counts,
    yerr=det_stds,
    color=[color_map[n] for n in bench_df["model_name"]],
    capsize=5,
    edgecolor="white",
    error_kw={"elinewidth": 1.5, "ecolor": "black"},
)
axes[1].set_ylabel("Mean detections per frame", fontsize=12)
axes[1].set_title(
    "Detection Count per Frame\n(20 test frames, proxy for sensitivity)",
    fontsize=12, fontweight="bold"
)
axes[1].set_xlabel("Model variant", fontsize=12)
axes[1].tick_params(axis="x", rotation=10)
axes[1].grid(axis="y", alpha=0.4)

# Annotation: this is a proxy, not mAP
axes[1].text(
    0.5, -0.2,
    "Note: Detection count is a proxy metric, not ground-truth mAP.",
    ha="center", transform=axes[1].transAxes,
    fontsize=8, color="grey", style="italic"
)

plt.suptitle(
    "YOLOv8 Model Comparison — Speed–Accuracy–Size Trade-off",
    fontsize=14, fontweight="bold", y=1.02
)
plt.tight_layout()
save_figure(fig, "07_tradeoff_scatter")
plt.show()

  Saved: reports\figures\07_tradeoff_scatter.png
  Saved: reports\figures\07_tradeoff_scatter.pdf
  Saved: paper\figures\07_tradeoff_scatter.png
  Saved: paper\figures\07_tradeoff_scatter.pdf


In [ ]:
# ── Final model recommendation table ──────────────────────────────────

# Added a deployment recommendation column based on use case
def recommend(row):
    if row["mean_fps"] >= 15:
        return "Edge / CPU real-time"
    elif row["mean_fps"] >= 8:
        return "Server / near real-time"
    else:
        return "Batch / offline processing"

bench_df["deployment_scenario"] = bench_df.apply(recommend, axis=1)

final_table = bench_df[[
    "model_name", "params_M", "model_size_mb",
    "mean_fps", "mean_latency_ms", "p95_latency_ms",
    "peak_memory_mb", "mean_det_count", "deployment_scenario"
]].copy()
final_table.columns = [
    "Model", "Params (M)", "Size (MB)",
    "Mean FPS", "Mean Lat (ms)", "p95 Lat (ms)",
    "Peak Mem (MB)", "Mean Dets/Frame", "Deployment Scenario"
]

print("Final model comparison table:")
display(final_table)

save_table(final_table, "07_final_model_comparison")
print("\n✓ Final table saved")

Final model comparison table:


,Model,Params (M),Size (MB),Mean FPS,Mean Lat (ms),p95 Lat (ms),Peak Mem (MB),Mean Dets/Frame,Deployment Scenario
0,YOLOv8-nano,3.15720,NaN,9.58,104.35,124.30,1.26,7.40,Server / near real-time
1,YOLOv8-small,11.16656,NaN,4.37,229.06,275.96,4.98,6.95,Batch / offline processing
2,YOLOv8-medium,25.90264,NaN,1.20,831.70,1388.52,0.48,8.40,Batch / offline processing


  Saved: reports\tables\07_final_model_comparison.csv
  Saved: reports\tables\07_final_model_comparison.tex
  Saved: paper\tables\07_final_model_comparison.csv
  Saved: paper\tables\07_final_model_comparison.tex

✓ Final table saved
